# Kapitan 06: validation and testing

What can break: the inventory (typos, unused classes, plaintext secrets), the render (a generator
change), the output (invalid Kubernetes objects), and the promise that `compiled/` matches the
sources. Each gets a check.


In [ ]:
cd /source/work/kapitan-reference
export HOME=/tmp
kapitan lint --skip-yamllint 2>&1 | tail -8 || true


In [ ]:
cd /source/work/kapitan-reference
kapitan compile 2>&1 | tail -6


Every target compiled with the kapitan in this image. `compiled/` is committed upstream, so `git status` is the rendered manifests check: a source change must ship with its output, and a tool upgrade shows exactly what it changed.


In [ ]:
cd /source/work/kapitan-reference
git status --short compiled | head -8; echo; git diff --stat compiled | tail -3


In [ ]:
cd /source/work/kapitan-reference
kubeconform -strict -ignore-missing-schemas -summary compiled/tutorial/manifests/*.yml compiled/mysql/manifests/*.yml compiled/hello/manifests/*.yml


Environment drift is a diff between two inventories, before anything is rendered.


In [ ]:
cd /source/work/kapitan-reference
diff <(kapitan inventory -t dev-sockshop -p parameters.components 2>/dev/null) <(kapitan inventory -t prod-sockshop -p parameters.components 2>/dev/null) | head -20 || true


Upstream CI goes one step further (`.github/workflows/integration-test.yml`): compile, create a kind cluster with the rendered `setup_cluster` script, apply the tutorial target, wait for the rollout. That is the integration test of a config model: it deploys.


In [ ]:
cd /source/work/kapitan-reference
sed -n 20,40p .github/workflows/integration-test.yml


In [ ]:
cd /source/work/kapitan-reference
git checkout -q compiled inventory 2>/dev/null; rm -f inventory/targets/tutorials/hello.yml; rm -rf compiled/hello system/refs/targets/hello; git status --short | head -3; echo cleaned
